# Faithful reproduction: patient-reported drug side-effect severity

This notebook reproduces the undergraduate thesis pipeline with the official UCI Drug Reviews (Druglib.com) dataset (ID 461; DOI: 10.24432/C55G6J). It is **not** the Kaggle Drugs.com dataset cited by the thesis.

Core design retained: spaCy lemmatization, TF-IDF (7500, 1-2 grams, min_df=5, max_df=0.95), one-hot effectiveness/rating/condition/urlDrugName, three binned classes, RF/LR/SVC grids, cv=3, random_state=42, balanced class weights.

In [ ]:
!pip -q install spacy==3.8.7 scikit-learn==1.7.2 seaborn wordcloud
!python -m spacy download en_core_web_sm -q
import json, os, random, time, urllib.request, zipfile
from pathlib import Path
import numpy as np, pandas as pd, spacy
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, label_binarize
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
SEED=42; random.seed(SEED); np.random.seed(SEED)
DATA_DIR=Path('/kaggle/working/data/raw'); DATA_DIR.mkdir(parents=True,exist_ok=True)
archive=DATA_DIR/'uci_druglib.zip'
if not (DATA_DIR/'drugLibTrain_raw.tsv').exists():
    urllib.request.urlretrieve('https://archive.ics.uci.edu/static/public/461/drug+review+dataset+druglib+com.zip',archive)
    with zipfile.ZipFile(archive) as z: z.extractall(DATA_DIR)
OUT=Path('/kaggle/working/results'); OUT.mkdir(exist_ok=True)


In [ ]:
train=pd.read_csv(DATA_DIR/'drugLibTrain_raw.tsv', sep='\t')
test=pd.read_csv(DATA_DIR/'drugLibTest_raw.tsv', sep='\t')
audit={'train_rows':len(train),'test_rows':len(test),'total_rows':len(train)+len(test),'columns':list(train.columns),'train_sideEffects_counts':train.sideEffects.value_counts().to_dict(),'test_sideEffects_counts':test.sideEffects.value_counts().to_dict(),'train_nulls':train.isna().sum().to_dict(),'test_nulls':test.isna().sum().to_dict()}
json.dump(audit,open(OUT/'dataset_audit.json','w'),indent=2)
audit


In [ ]:
nlp=spacy.load('en_core_web_sm')
labels=['No Side Effects','Mild Side Effects','Severe Side Effects']; cats=['effectiveness','rating','condition','urlDrugName']
def bin_effect(x):
    x=str(x)
    if 'No Side Effects' in x: return 'No Side Effects'
    if 'Mild Side Effects' in x or 'Moderate Side Effects' in x: return 'Mild Side Effects'
    return 'Severe Side Effects'
def prep(df):
    df=df.copy(); text=df.benefitsReview.fillna('')+' '+df.commentsReview.fillna('')+' '+df.sideEffectsReview.fillna('')
    df['processed_reviews']=[' '.join(t.lemma_ for t in d if t.is_alpha and not t.is_stop) for d in nlp.pipe(text.astype(str).str.lower(),batch_size=64)]
    df['target']=df.sideEffects.apply(bin_effect); return df
train,test=prep(train),prep(test)
tfidf=TfidfVectorizer(max_features=7500,ngram_range=(1,2),min_df=5,max_df=.95)
enc=OneHotEncoder(handle_unknown='ignore')
Xtr=hstack([tfidf.fit_transform(train.processed_reviews),enc.fit_transform(train[cats].astype(str))])
Xte=hstack([tfidf.transform(test.processed_reviews),enc.transform(test[cats].astype(str))])
ytr,yte=train.target,test.target
print('Feature shape:',Xtr.shape); print(ytr.value_counts()); print(yte.value_counts())


In [ ]:
grids={'Logistic Regression':GridSearchCV(LogisticRegression(random_state=42,class_weight='balanced',max_iter=1000),{'C':[1,10,20],'solver':['saga'],'penalty':['l1','l2']},cv=3,n_jobs=-1,scoring='accuracy'),'Random Forest':GridSearchCV(RandomForestClassifier(random_state=42,class_weight='balanced',n_jobs=-1),{'n_estimators':[200,300],'max_depth':[20,30],'min_samples_split':[2,5],'min_samples_leaf':[1,2]},cv=3,n_jobs=-1,scoring='accuracy'),'SVC':GridSearchCV(SVC(random_state=42,class_weight='balanced',gamma='scale'),{'C':[1,10,50],'kernel':['linear','rbf']},cv=3,n_jobs=-1,scoring='accuracy')}
models={}; results={}
for name,g in grids.items():
    g.fit(Xtr,ytr); p=g.predict(Xte); models[name]=g.best_estimator_
    results[name]={'best_params':g.best_params_,'cv_accuracy':g.best_score_,'test_accuracy':accuracy_score(yte,p),'classification_report_corrected':classification_report(yte,p,labels=labels,target_names=labels,output_dict=True),'confusion_matrix_label_order_No_Mild_Severe':confusion_matrix(yte,p,labels=labels).tolist()}
json.dump(results,open(OUT/'official_split_corrected_results.json','w'),indent=2,default=float)
pd.DataFrame({'model':results.keys(),'accuracy':[x['test_accuracy'] for x in results.values()]}).sort_values('accuracy',ascending=False)


## Integrity disclosure
The appendix fits the vectorizer and encoder once before CV. Therefore GridSearchCV and learning-curve CV see features fit on the whole training split. This notebook retains that for faithful reproduction; a Pipeline-within-fold analysis must be reported separately as supplementary, not substituted for the headline result.

The original appendix also mislabels classification-report class rows because sklearn sorts labels alphabetically, and misaligns ROC probability columns. The saved output above corrects both reporting defects explicitly.

In [ ]:
review_cols=['benefitsReview','commentsReview','sideEffectsReview']
norm=lambda s:' '.join(str(s).lower().split())
tr_text=train[review_cols].fillna('').agg(' '.join,axis=1).map(norm); te_text=test[review_cols].fillna('').agg(' '.join,axis=1).map(norm)
integrity={'exact_combined_review_overlap_test_rows':int(te_text.isin(set(tr_text)).sum()),'exact_overlap_by_field':{c:int(test[c].fillna('').map(norm).isin(set(train[c].fillna('').map(norm))).sum()) for c in review_cols},'note':'Exact overlap is diagnostic only; do not replace official-split headline results without author decision.'}
json.dump(integrity,open(OUT/'integrity_checks.json','w'),indent=2); integrity


## Textual analyses reproduced from the thesis
These cells recreate the class-distribution figure, processed-review length histogram, training-review word cloud, and top-term table. The thesis code uses `len(processed_reviews)`, so the length plot is correctly labelled character length rather than word count. All figures and tables are saved under `figures/` and `results/`.


In [ ]:
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from wordcloud import WordCloud
import matplotlib.pyplot as plt, seaborn as sns
FIG=Path('/kaggle/working/figures'); FIG.mkdir(exist_ok=True)
raw_counts=train['sideEffects'].value_counts().rename_axis('sideEffects').reset_index(name='count')
binned_counts=train['target'].value_counts().rename_axis('target').reset_index(name='count')
classes=np.array(sorted(train['target'].unique())); weights=dict(zip(classes,compute_class_weight(class_weight='balanced',classes=classes,y=train['target'])))
binned_counts['class_weight']=binned_counts['target'].map(weights); raw_counts.to_csv(OUT/'textual_raw_side_effect_counts.csv',index=False); binned_counts.to_csv(OUT/'textual_binned_counts_weights.csv',index=False)
plt.figure(figsize=(9,5)); ax=sns.barplot(data=binned_counts,x='target',y='count',order=labels,palette='viridis')
for i,cl in enumerate(labels): ax.text(i,binned_counts.set_index('target').loc[cl,'count']+8,f'weight={weights[cl]:.2f}',ha='center')
plt.title('Binned side-effect distribution with balanced class weights'); plt.xlabel('Side-effect severity'); plt.ylabel('Training reviews'); plt.tight_layout(); plt.savefig(FIG/'side_effect_distribution_with_weights.png',dpi=300); plt.show()
train['processed_review_length']=train['processed_reviews'].str.len(); json.dump(train['processed_review_length'].describe().to_dict(),open(OUT/'textual_processed_review_length_summary.json','w'),indent=2)
plt.figure(figsize=(10,6)); sns.histplot(train['processed_review_length'],bins=50,kde=True); plt.title('Distribution of processed review character lengths'); plt.xlabel('Processed review length (characters)'); plt.ylabel('Frequency'); plt.tight_layout(); plt.savefig(FIG/'processed_review_character_length.png',dpi=300); plt.show()
corpus=' '.join(train['processed_reviews'].fillna('')); wc=WordCloud(width=1600,height=800,background_color='white',collocations=False,random_state=42).generate(corpus)
plt.figure(figsize=(16,8)); plt.imshow(wc,interpolation='bilinear'); plt.axis('off'); plt.title('Word cloud of processed training reviews'); plt.tight_layout(); plt.savefig(FIG/'wordcloud_train_reviews.png',dpi=300,bbox_inches='tight'); plt.show()
top=pd.DataFrame(Counter(corpus.split()).most_common(30),columns=['term','count']); top.to_csv(OUT/'textual_top_terms.csv',index=False)
plt.figure(figsize=(10,8)); sns.barplot(data=top.head(20),y='term',x='count',color='#4c72b0'); plt.title('Top terms in processed training reviews'); plt.tight_layout(); plt.savefig(FIG/'top_terms_train_reviews.png',dpi=300); plt.show()


## Confusion matrices
These reproduce the appendix confusion-matrix section using the official UCI test split. The explicit label order is `[No Side Effects, Mild Side Effects, Severe Side Effects]`; each matrix is printed and saved as both PNG and CSV.


In [ ]:
from sklearn.metrics import confusion_matrix
CM_DIR=Path('/kaggle/working/figures'); CM_DIR.mkdir(exist_ok=True)
cm_results={}
for name,model in models.items():
    pred=model.predict(Xte); cm=confusion_matrix(yte,pred,labels=labels); cm_results[name]=cm.tolist()
    pd.DataFrame(cm,index=labels,columns=labels).to_csv(OUT/f'confusion_matrix_{name.replace(" ","_")}.csv')
    plt.figure(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=labels,yticklabels=labels,cbar=False)
    plt.title(f'Confusion Matrix: {name} (official UCI test split)'); plt.xlabel('Predicted label'); plt.ylabel('True label'); plt.tight_layout(); plt.savefig(CM_DIR/f'confusion_matrix_{name.replace(" ","_")}.png',dpi=300,bbox_inches='tight'); plt.show()
print('Confusion matrices, label order =',labels); print(json.dumps(cm_results,indent=2)); print('Saved:',sorted(p.name for p in CM_DIR.glob('confusion_matrix_*.png')))
